# FixForward — Repair-or-Replace Decision Agent

**Hackathon demo notebook**

FixForward helps structure repair-versus-replacement assessments for electric fans and mixer grinders. It combines deterministic safety screening and cost comparison with a tool-selection decision and an optional Amazon Bedrock explanation.

> **Important:** A local heuristic fallback is not Strands Decider inference. This notebook labels the decision mode and only reports live Decider/Bedrock use when the configured services respond successfully. Use verified manuals before making manufacturer-specific claims.

## 1. Environment and imports

Run this notebook from the root of the FixForward repository (the folder containing `agent/`, `tools/`, `services/`, and `app.py`). Install dependencies first using the repository's `requirements.txt`.

In [ ]:
import os, json, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'agent').exists():
    raise RuntimeError('Open this notebook from the FixForward project root, or change PROJECT_ROOT to that folder.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from tools.input_validator import validate_case
from tools.safety_screen import screen_safety
from tools.cost_engine import compare_costs
from agent.decider_router import choose_action
from services.bedrock_explainer import explain_assessment

print('Project root:', PROJECT_ROOT)
print('DECIDER_URL configured:', bool(os.getenv('DECIDER_URL')))
print('BEDROCK_MODEL_ID configured:', bool(os.getenv('BEDROCK_MODEL_ID')))


## 2. Example appliance case

These are illustrative user-supplied estimates, not market prices or a manufacturer diagnosis. Change them to test other scenarios.

In [ ]:
case = {
    'category': 'electric_fan',
    'age_years': 3,
    'symptoms': ['intermittent_operation'],
    'warranty_status': 'unknown',
    'repair_quote_inr': 700,
    'replacement_price_inr': 2200,
    'notes': 'Illustrative scenario for demonstrating the workflow.'
}
print(json.dumps(case, indent=2, ensure_ascii=False))

## 3. Validate the input

Validation checks that the supplied case has the expected structure. It does not diagnose the appliance.

In [ ]:
validation = validate_case(case)
print(json.dumps(validation, indent=2, ensure_ascii=False))

## 4. Apply the safety gate first

The deterministic safety gate flags reported burning smell, smoke/sparks, or overheating. A negative result does not certify the appliance as safe.

In [ ]:
safety = screen_safety(case)
print(json.dumps(safety, indent=2, ensure_ascii=False))

if safety['hazard']:
    print('\nSAFETY OVERRIDE: stop cost-based recommendation and advise qualified assistance.')
else:
    print('\nNo predefined hazard symptom was reported; this is not a safety certification.')

## 5. Strands Decider 2B tool selection

If `DECIDER_URL` points to a reachable Strands Decider HTTP service, this cell attempts real inference. Otherwise, the project's router returns a heuristic demo decision. Always inspect the `mode` field before describing results as model inference.

In [ ]:
decision = choose_action(case)
print(json.dumps(decision, indent=2, ensure_ascii=False))

if decision.get('mode') != 'strands_decider_2b':
    print('\nNOTICE: This is not a live Strands Decider 2B result. Configure DECIDER_URL and rerun to test the real endpoint.')
else:
    print('\nLive Strands Decider endpoint responded successfully.')

## 6. Run the deterministic cost comparison

This compares only the user-provided upfront repair and replacement estimates. It does not model future maintenance, downtime, warranty value, or remaining service life.

In [ ]:
cost_comparison = compare_costs(case)
print(json.dumps(cost_comparison, indent=2, ensure_ascii=False))

## 7. Generate the assessment explanation with Amazon Bedrock

When `BEDROCK_MODEL_ID` and valid AWS credentials/permissions are configured, the project attempts a real Bedrock Converse API call. Otherwise, it returns a deterministic fallback explanation. Inspect the output note and Render/AWS logs to confirm the actual path.

In [ ]:
assessment = {
    'case': case,
    'safety': safety,
    'decision': decision,
    'cost_comparison': cost_comparison,
    'evidence': []
}
explanation = explain_assessment(assessment)
print(explanation)

if not os.getenv('BEDROCK_MODEL_ID'):
    print('\nNOTICE: BEDROCK_MODEL_ID is not configured; this is a deterministic fallback, not a Bedrock response.')

## 8. Safety regression test

This example verifies that a known hazardous symptom is flagged by the deterministic safety gate.

In [ ]:
hazard_case = {**case, 'symptoms': ['smoke_or_sparks']}
hazard_result = screen_safety(hazard_case)
assert hazard_result['hazard'] is True
assert 'smoke_or_sparks' in hazard_result['flags']
print('PASS — smoke/sparks triggers the safety gate.')
print(json.dumps(hazard_result, indent=2, ensure_ascii=False))

## 9. Missing-information test

This test checks that the workflow can identify a case where price inputs are absent. The selected action may be a live Decider result or a clearly labelled fallback.

In [ ]:
incomplete_case = {
    'category': 'mixer_grinder',
    'age_years': 2,
    'symptoms': ['intermittent_operation'],
    'warranty_status': 'unknown',
    'repair_quote_inr': None,
    'replacement_price_inr': None,
    'notes': 'Missing cost information.'
}
incomplete_decision = choose_action(incomplete_case)
print(json.dumps(incomplete_decision, indent=2, ensure_ascii=False))

## 10. Limitations and reproducibility

- A heuristic fallback must never be represented as Strands Decider inference.
- A deterministic explanation must never be represented as an Amazon Bedrock response.
- The sample reference data is illustrative; replace it with verified documents before making manufacturer-specific claims.
- User-provided cost estimates may be inaccurate or incomplete.
- The safety gate catches only predefined symptoms and is not a certification that an appliance is safe.
- Do not provide DIY instructions for opening or repairing potentially hazardous electrical equipment.

### How to reproduce

1. Clone the public FixForward repository.
2. Install `requirements.txt` in a Python environment.
3. Run this notebook from the repository root.
4. For real model tests, configure a reachable `DECIDER_URL`, `BEDROCK_MODEL_ID`, `AWS_REGION`, and secure AWS credentials with permission to invoke the selected model.
5. Record whether each run used live inference or fallback behavior.